# Mass Balance and Closure Test Notebook

This notebook tests mass-balance reporting, closure calculations, and export integration.

Coverage:
- projected runs still report mass balance on the actual simulated window
- mass-balance component rows are consistent with `simulated_*` summary stats
- closure rows match the component-based mass conversions
- lactate and ammonium production follow `out - in`
- Excel export includes the mass-balance sheet

## 1. Setup

In [ ]:
import sys
import tempfile
import shutil
import importlib
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))


from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.gas_controllers import GasController
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.perfusion_strategies import PerfusionStrategy


def get_row(df, column, value):
    row = df.loc[df[column] == value]
    assert len(row) == 1, f'Expected one row for {value!r}'
    return row.iloc[0]

def assert_close(actual, expected, label, rtol=1e-9, atol=1e-9):
    assert np.isclose(actual, expected, rtol=rtol, atol=atol), (
        f'{label}: expected {expected:.12g}, got {actual:.12g}'
    )

def mmol_to_kg(mmol, mw):
    return mmol / 1000.0 * mw / 1000.0

test_export_dir = Path(tempfile.mkdtemp(prefix='animakinetica_mass_balance_'))
test_export_dir = Path('C:/Users/btumulero/OneDrive - Delft University of Technology/Documents/01. Code/animakinetica/test/exports') # Override for saving locally instead of in temp directory

print('✓ Setup complete')

✓ Setup complete


## 2. Run a Projected Simulation

In [ ]:
params = SimulationParameters.load_parameters(['Cell_GFI', 'anab_GFI', 'cata_GFI', 'fed_batch'])
params.update("perfusion")
params.update("excess_kinetics_batch")
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    stoichiometry_mode=StoichiometryMode.DYNAMIC,
    operating_mode=OperatingMode.FEDBATCH,
    feed_strategy=FeedStrategy.CONSTANT,
    #perfusion_strategy=PerfusionStrategy.CSPR,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,
    solver_method='BDF',
)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Perfusion Strategy: {sim.perfusion_strategy}")
print(f"  Gas Controller: {sim.gas_controller}")
print(f"  Bleed Rate (B): {sim.parameters.perfusion.B.value:.4f} 1/h")


Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).
✓ Simulation initialized
  Operating Mode: Fedbatch
  Kinetics Model: Excess
  

In [10]:
# Run Simulation
num_batches = 10
data = sim.run_simulation(number_of_batches=num_batches)

projected_totals_stats = data.projected_totals_stats
simulated_totals_stats = data.simulated_window_totals_stats

print(f"projection_mode={projected_totals_stats['projection_mode']}")
print(f"projection_scale_factor={projected_totals_stats['projection_scale_factor']:.6f}")

assert 'total_DCM_out_kg' in projected_totals_stats
print(f"Projected DCM out: {projected_totals_stats['total_DCM_out_kg']:.6f} kg")
assert projected_totals_stats['projection_scale_factor'] >= 1.0
if projected_totals_stats['projection_scale_factor'] > 1.0:
    assert not np.isclose(projected_totals_stats['total_DCM_out_kg'], simulated_totals_stats['total_DCM_out_kg'])

print('✓ Projected simulation complete')

run_simulation execution time: 906.960 ms
projection_mode=full_simulation
projection_scale_factor=1.000000
Projected DCM out: 6.312079 kg
✓ Projected simulation complete


## 3. CHON Elemental Closure

This section now shows two different process-level CHON diagnostics:

- `compute_stoichiometric_closure()` integrates the simulated `mu` and `q_*` trajectories over the stored batch data. This is the stoichiometric process-closure check and should close tightly when the reaction basis is internally consistent.
- `compute_boundary_closure()` builds a whole-process external boundary balance from simulated-window inputs and outputs only. It excludes internal carryover between repeated batches, uses the corrected batch-boundary bookkeeping, and uses trapezoidal flow integration so the black-box process balance is materially tighter than the older reduced-flow diagnostic.
- `compute_closure_delta()` calculates the difference between the two previously mentioned CHON closures.

### 3.1 Stoichiometric closure

In [11]:
process_stoich_closure = data.compute_stoichiometric_closure()
process_stoich_closure_df = data.build_closure_summary_df(process_stoich_closure)

process_stoich_biomass = next(
    spec for spec in process_stoich_closure['component_specs']
    if spec['label'] == 'Dry biomass'
)

assert process_stoich_closure['basis'] == 'integrated_stoichiometric_process_chon_closure'
assert list(process_stoich_closure_df['Element']) == ['C', 'H', 'O', 'N']
assert np.isfinite(process_stoich_closure['max_abs_residual'])
assert np.all(np.isfinite(process_stoich_closure_df['Residual (mol-atoms)']))
assert process_stoich_biomass['amount'] > 0.0

print('Integrated stoichiometric CHON closure:')
display(process_stoich_closure_df)
print(f"Max absolute elemental residual: {process_stoich_closure['max_abs_residual']:.6g} mol-atoms")
print('Stoichiometric component basis:')
display(pd.DataFrame(process_stoich_closure['component_specs']))


Integrated stoichiometric CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,1230.808524,1230.798219,-0.010305,-0.000837
1,H,2472.513600,2472.489498,-0.024103,-0.000975
2,O,1847.726101,1847.719283,-0.006818,-0.000369
3,N,143.531243,143.527754,-0.003489,-0.002431


Max absolute elemental residual: 0.0241029 mol-atoms
Stoichiometric component basis:


,label,side,description,amount,amount_unit,formula,yield_on_biomass,yield_unit
0,Glucose consumed,Input,Integrated glucose uptake from the simulated s...,142.032065,mol,"{'C': 6.0, 'H': 12.0, 'O': 6.0, 'N': 0.0}",0.498482,mol / Cmol DCM
1,Glutamine consumed,Input,Integrated glutamine uptake from the simulated...,55.652525,mol,"{'C': 5.0, 'H': 10.0, 'O': 3.0, 'N': 2.0}",0.195321,mol / Cmol DCM
2,Amino acids consumed,Input,Integrated uptake of the lumped amino-acid poo...,21.196578,mol,"{'C': 4.734420367155564, 'H': 9.98291200026723...",0.074392,mol / Cmol DCM
3,O2 consumed,Input,Integrated oxygen uptake from the simulated sp...,398.219154,mol,"{'C': 0.0, 'H': 0.0, 'O': 2.0, 'N': 0.0}",1.397608,mol / Cmol DCM
4,Dry biomass,Output,Integrated biomass formation from the simulate...,284.929132,Cmol,"{'C': 1.0, 'H': 1.64100082296572, 'O': 0.32346...",1.000000,Cmol / Cmol DCM
5,Lactate produced,Output,Integrated lactate formation from the simulate...,169.184556,mol,"{'C': 3.0, 'H': 6.0, 'O': 3.0, 'N': 0.0}",0.593778,mol / Cmol DCM
6,Ammonium produced,Output,Integrated ammonium formation from the simulat...,82.356838,mol,"{'C': 0.0, 'H': 3.0, 'O': 0.0, 'N': 1.0}",0.289043,mol / Cmol DCM
7,CO2 produced,Output,Integrated carbon dioxide formation from the s...,438.315419,mol,"{'C': 1.0, 'H': 0.0, 'O': 2.0, 'N': 0.0}",1.538331,mol / Cmol DCM
8,H2O produced,Output,Integrated metabolic water formation from the ...,371.371353,mol,"{'C': 0.0, 'H': 2.0, 'O': 1.0, 'N': 0.0}",1.303381,mol / Cmol DCM


### 3.2 Process Boundary Closure

In [12]:
process_external_elemental_closure = data.compute_boundary_closure()
process_external_elemental_closure_df = data.build_closure_summary_df(process_external_elemental_closure)

process_dry_biomass = next(
    spec for spec in process_external_elemental_closure['component_specs']
    if spec['label'] == 'DCM out'
 )

assert_close(
    process_dry_biomass['amount'],
    simulated_totals_stats['total_DCM_out_cmmol'] / 1000,
    'process external elemental biomass basis',
)
assert list(process_external_elemental_closure_df['Element']) == ['C', 'H', 'O', 'N']
assert np.isfinite(process_external_elemental_closure['max_abs_residual'])
assert abs(process_external_elemental_closure['elements']['H']['relative_residual_pct']) < 0.1
assert abs(process_external_elemental_closure['elements']['O']['relative_residual_pct']) < 0.1
assert abs(process_external_elemental_closure['elements']['C']['relative_residual_pct']) < 2.0
assert abs(process_external_elemental_closure['elements']['N']['relative_residual_pct']) < 4.0
assert np.all(np.isfinite(process_external_elemental_closure_df['Residual (mol-atoms)']))

print('Whole-process external-boundary CHON closure:')
display(process_external_elemental_closure_df)
print(f"Max absolute elemental residual: {process_external_elemental_closure['max_abs_residual']:.6g} mol-atoms")
print('External boundary species amounts:')
display(pd.DataFrame(process_external_elemental_closure['component_specs']))


Whole-process external-boundary CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,2.237935e+03,2.237971e+03,0.036007,0.001609
1,H,1.125610e+06,1.125610e+06,0.032464,0.000003
2,O,5.636323e+05,5.636323e+05,0.007204,0.000001
3,N,3.565308e+02,3.565273e+02,-0.003490,-0.000979


Max absolute elemental residual: 0.0360073 mol-atoms
External boundary species amounts:


,label,side,description,amount,amount_unit,formula
0,Fresh glucose in,Input,Total glucose entering with fresh initial char...,1.595750e+02,mol,"{'C': 6.0, 'H': 12.0, 'O': 6.0, 'N': 0.0}"
1,Fresh glutamine in,Input,Total glutamine entering with fresh initial ch...,1.010359e+02,mol,"{'C': 5.0, 'H': 10.0, 'O': 3.0, 'N': 2.0}"
2,Fresh amino acids in,Input,Total lumped amino-acid pool entering with fre...,1.010359e+02,mol,"{'C': 4.734420367155564, 'H': 9.98291200026723..."
3,Fresh lactate in,Input,Lactate entering from fresh external media only.,2.273737e-16,mol,"{'C': 3.0, 'H': 6.0, 'O': 3.0, 'N': 0.0}"
4,Fresh ammonium in,Input,Ammonium entering from fresh external media only.,0.000000e+00,mol,"{'C': 0.0, 'H': 3.0, 'O': 0.0, 'N': 1.0}"
5,O2 in,Input,Total oxygen crossing the reactor boundary int...,3.983202e+02,mol,"{'C': 0.0, 'H': 0.0, 'O': 2.0, 'N': 0.0}"
6,CO2 in,Input,Total inorganic carbon entering from fresh med...,2.930042e+02,mol,"{'C': 1.0, 'H': 0.0, 'O': 2.0, 'N': 0.0}"
7,"Medium, base, and acid water in",Input,Carrier water entering with fresh medium charg...,5.608347e+05,mol,"{'C': 0.0, 'H': 2.0, 'O': 1.0, 'N': 0.0}"
8,DCM in,Input,"Dry biomass entering via inoculum, only with f...",3.954737e+00,Cmol,"{'C': 1.0, 'H': 1.64100082296572, 'O': 0.32346..."
9,Glucose out,Output,Total glucose leaving in external liquid outle...,1.754103e+01,mol,"{'C': 6.0, 'H': 12.0, 'O': 6.0, 'N': 0.0}"


### 3.3 Delta Stoichiometric and Process Boundary Closure

In [13]:
delta_result = data.compute_closure_delta()
delta_df = data.get_closure_delta_df()
display(delta_df)

for closure, label in ((process_stoich_closure, 'stoichiometric'), (process_external_elemental_closure, 'boundary')):
    assert list(data.build_closure_summary_df(closure)['Element']) == ['C', 'H', 'O', 'N']
    assert np.isfinite(closure['max_abs_residual']), f'{label} closure residual is not finite'

# Sanity check: boundary DCM-out basis agrees with the simulated-window totals.
boundary_dcm_out = next(spec for spec in process_external_elemental_closure['component_specs'] if spec['label'] == 'DCM out')
assert_close(
    boundary_dcm_out['amount'],
    simulated_totals_stats['total_DCM_out_cmmol'] / 1000.0,
    'boundary DCM-out basis vs simulated totals',
)


,Component,Side,Description,Stoich Amount,Boundary Net Amount,Delta Amount,Amount Unit,C,H,O,N
0,Glucose consumed delta,Input,Difference between boundary net flux and integ...,142.032065,142.033973,1.908075e-03,mol,6.00000,12.000000,6.000000,0.000000
1,Glutamine consumed delta,Input,Difference between boundary net flux and integ...,55.652525,55.648040,-4.484962e-03,mol,5.00000,10.000000,3.000000,2.000000
2,Amino acids consumed delta,Input,Difference between boundary net flux and integ...,21.196578,21.199201,2.622855e-03,mol,4.73442,9.982912,1.516180,1.520349
3,O2 consumed delta,Input,Difference between boundary net flux and integ...,398.219154,398.219208,5.441134e-05,mol,0.00000,0.000000,2.000000,0.000000
4,Dry biomass delta,Output,Difference between boundary net flux and integ...,284.929132,284.964386,3.525374e-02,Cmol,1.00000,1.641001,0.323461,0.214688
5,Lactate produced delta,Output,Difference between boundary net flux and integ...,169.184556,169.191323,6.767303e-03,mol,3.00000,6.000000,3.000000,0.000000
6,Ammonium produced delta,Output,Difference between boundary net flux and integ...,82.356838,82.344286,-1.255196e-02,mol,0.00000,3.000000,0.000000,1.000000
7,CO2 produced delta,Output,Difference between boundary net flux and integ...,438.315419,438.307617,-7.801888e-03,mol,1.00000,0.000000,2.000000,0.000000
8,H2O produced delta,Output,Difference between boundary net flux and integ...,371.371353,371.371352,-7.445766e-07,mol,0.00000,2.000000,1.000000,0.000000


## 4. Export Integration

The Excel export should now distinguish the three reporting strategies explicitly:

- `Operational Mass Balance`: simulated-window boundary totals plus the reduced mass closure summary.
- `Stoich CHON Closure`: process-level CHON closure integrated from stored `mu` and `q_*` trajectories.
- `Boundary CHON Closure`: whole-process external-boundary CHON closure from simulated-window inputs and outputs only.

In [16]:
export_path = data.save(
    filename='mass_balance_test_export',
    file_format='excel',
    add_timestamp=False,
    include_metadata=True,
    include_summary_stats=True,
    include_mass_balance=True,
    include_cultivation_summary=True,
    include_batch_data=False,
    include_parameters=False,
    include_pickle=False,
    overwrite=True,
    export_dir=test_export_dir,
)

assert Path(export_path).exists()


stoich_sheet = 'Stoich CHON Closure'
boundary_sheet = 'Boundary CHON Closure'
delta_sheet = 'CHON Closure Delta'
stoich_component_rows = len(process_stoich_closure['component_specs'])
boundary_component_rows = len(process_external_elemental_closure['component_specs'])
delta_component_rows = len(delta_result['component_specs'])

with pd.ExcelFile(export_path) as xls:
    assert stoich_sheet in xls.sheet_names
    assert boundary_sheet in xls.sheet_names
    assert delta_sheet in xls.sheet_names

    exported_stoich_closure_df = pd.read_excel(
        xls,
        sheet_name=stoich_sheet,
        skiprows=1,
        nrows=len(process_stoich_closure_df.index),
    )
    exported_stoich_components_df = pd.read_excel(
        xls,
        sheet_name=stoich_sheet,
        skiprows=len(process_stoich_closure_df.index) + 5,
        nrows=stoich_component_rows,
    )
    exported_stoich_formula_df = pd.read_excel(
        xls,
        sheet_name=stoich_sheet,
        skiprows=len(process_stoich_closure_df.index) + stoich_component_rows + 9,
        nrows=2,
    )

    exported_boundary_closure_df = pd.read_excel(
        xls,
        sheet_name=boundary_sheet,
        skiprows=1,
        nrows=len(process_external_elemental_closure_df.index),
    )
    exported_boundary_components_df = pd.read_excel(
        xls,
        sheet_name=boundary_sheet,
        skiprows=len(process_external_elemental_closure_df.index) + 5,
        nrows=boundary_component_rows,
    )
    exported_boundary_formula_df = pd.read_excel(
        xls,
        sheet_name=boundary_sheet,
        skiprows=len(process_external_elemental_closure_df.index) + boundary_component_rows + 9,
        nrows=2,
    )

    # First section of the delta sheet is the 4-row element-level summary table
    # (Element, Stoich Reactants, Stoich Products, Stoich Residual, Boundary...,
    # Delta Residual) -- NOT the 9-row per-component table (that is section 2).
    exported_delta_closure_df = pd.read_excel(
        xls,
        sheet_name=delta_sheet,
        skiprows=1,
        nrows=4,
    )

assert list(exported_stoich_closure_df['Element']) == ['C', 'H', 'O', 'N']
assert np.all(np.isfinite(exported_stoich_closure_df['Residual (mol-atoms)']))
assert 'Component' in exported_stoich_components_df.columns
assert 'Dry biomass' in set(exported_stoich_components_df['Component'])
assert 'Formula' in exported_stoich_formula_df.columns
assert 'Biomass CHON basis' in set(exported_stoich_formula_df['Formula'])

assert list(exported_boundary_closure_df['Element']) == ['C', 'H', 'O', 'N']
assert np.all(np.isfinite(exported_boundary_closure_df['Residual (mol-atoms)']))
assert 'Component' in exported_boundary_components_df.columns
assert 'DCM out' in set(exported_boundary_components_df['Component'])
assert 'Formula' in exported_boundary_formula_df.columns
assert 'Lumped amino-acid CHON basis' in set(exported_boundary_formula_df['Formula'])

assert list(exported_delta_closure_df['Element']) == ['C', 'H', 'O', 'N']
assert np.all(np.isfinite(exported_delta_closure_df['Delta Residual (mol-atoms)']))

print(f'✓ Export integration passed: {export_path}')

Simulation data exported to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\test\exports\mass_balance_test_export\mass_balance_test_export.xlsx
✓ Export integration passed: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\test\exports\mass_balance_test_export\mass_balance_test_export.xlsx


## 5. Cleanup

In [17]:
if test_export_dir.exists():
    shutil.rmtree(test_export_dir, ignore_errors=True)

print('✓ Cleanup complete')

✓ Cleanup complete
